In [ ]:
"""Item-specific encoding-retrieval similarity (H5), sliding-window version.

Feature vector = all channels x samples inside a window (Lu et al., 2015 style).
Per participant: D = mean_i[ z(match_i) - mean_{j != i} z(nonmatch_ij) ],
with z = Fisher z of the Pearson r between an encoding window and a retrieval window.
"""
import numpy as np
import pandas as pd


def get_hit_pairs(beh_csv):
    """Return DataFrame of hits (old cue answered old) with encoding trial id (1-based
    enc_trial_count) and retrieval trial id, plus condition columns."""
    d = pd.read_csv(beh_csv)
    h = d[(d.ret_trial_type == "old") & (d.cue_recog_response == "old") & (~d.bad_trial)].copy()
    h["enc_trial_count"] = h.enc_trial_count.astype(int)
    h["ret_trial_count"] = h.ret_trial_count.astype(int)
    return h[["enc_trial_count", "ret_trial_count", "enc_low_prediction",
              "enc_high_prediction", "enc_acc"]].reset_index(drop=True)


def make_windows(data, sfreq, tmin_epoch, win_ms=100, step_ms=20, t_start=0.0,
                 spatial_center=False):
    """data: (n_trials, n_channels, n_times) -> z-scored windows
    (n_trials, n_win, n_ch*win) and window-centre times (s).
    Windows start at t_start (so the baseline never enters a window)."""
    n_tr, n_ch, n_t = data.shape
    w = int(round(win_ms / 1000 * sfreq))
    step = int(round(step_ms / 1000 * sfreq))
    first = int(round((t_start - tmin_epoch) * sfreq))
    starts = np.arange(first, n_t - w + 1, step)
    if spatial_center:  # remove channel mean at every sample -> spatial-only patterns
        data = data - data.mean(axis=1, keepdims=True)
    idx = starts[:, None] + np.arange(w)[None, :]            # (n_win, w)
    win = data[:, :, idx]                                    # (n_tr, ch, n_win, w)
    win = win.transpose(0, 2, 1, 3).reshape(n_tr, len(starts), n_ch * w)
    win = win - win.mean(-1, keepdims=True)
    win = win / (win.std(-1, keepdims=True) + 1e-12)
    centres = tmin_epoch + (starts + w / 2) / sfreq
    return win, centres


def ers_difference(enc, ret, enc_idx, ret_idx, sfreq, enc_tmin=-0.2, ret_tmin=-0.2,
                   win_ms=100, step_ms=20, spatial_center=False,
                   return_parts=False):
    """enc: (n_enc_trials, ch, t) epochs in encoding-trial order;
    ret: (n_ret_trials, ch, t) retrieval epochs.
    enc_idx[k], ret_idx[k]: row indices into enc / ret for hit k (matching pair).
    Only hits whose BOTH epochs survived should be passed in.
    Returns D (n_enc_win, n_ret_win) and the window-centre times."""
    we, te = make_windows(enc[enc_idx], sfreq, enc_tmin, win_ms, step_ms, spatial_center=spatial_center)
    wr, tr = make_windows(ret[ret_idx], sfreq, ret_tmin, win_ms, step_ms, spatial_center=spatial_center)
    n, F = we.shape[0], we.shape[-1]
    match = np.zeros((len(te), len(tr)))
    nonmatch = np.zeros_like(match)
    for i in range(n):
        # r between retrieval i and every encoding stream j: (n, n_enc_win, n_ret_win)
        r = np.tensordot(we, wr[i].T, axes=([2], [0])) / F
        z = np.arctanh(np.clip(r, -0.999999, 0.999999))
        match += z[i]
        nonmatch += (z.sum(0) - z[i]) / (n - 1)
    D = (match - nonmatch) / n
    if return_parts:
        return D, te, tr, match / n, nonmatch / n
    return D, te, tr


def segment_means(D, te, tr, segments=None):
    """Mean D per encoding segment (descriptive only; test the full matrix)."""
    segments = segments or {"cue": (0, 2), "action1": (2, 3), "action2": (3, 4),
                            "action3": (4, 5), "target": (5, 7)}
    out = {}
    for k, (a, b) in segments.items():
        m = (te >= a) & (te < b)
        out[k] = D[m].mean(0)  # profile over retrieval time
    return out


def pair_hits_from_epochs(enc_epochs, ret_epochs):
    """Build the matching pairs from the *surviving* epochs' metadata.

    A hit = old cue answered old, bad_trial FALSE, and an encoding epoch with the same
    enc_trial_count survived. Returns (enc_idx, ret_idx, hits_df) where the indices are
    positions in enc_epochs / ret_epochs and hits_df holds the condition columns (aligned)."""
    enc_md, ret_md = enc_epochs.metadata, ret_epochs.metadata
    enc_pos = {int(k): i for i, k in enumerate(enc_md["enc_trial_count"])}
    is_hit = ((ret_md["ret_trial_type"] == "old") & (ret_md["cue_recog_response"] == "old")
              & (~ret_md["bad_trial"].astype(bool))).to_numpy()
    enc_idx, ret_idx = [], []
    for ri in np.flatnonzero(is_hit):
        k = int(ret_md["enc_trial_count"].iloc[ri])
        if k in enc_pos:
            enc_idx.append(enc_pos[k]); ret_idx.append(int(ri))
    hits = ret_md.iloc[ret_idx].reset_index(drop=True)
    return np.array(enc_idx), np.array(ret_idx), hits


def run_ers_subject(epo_dir, subject_id, win_ms=100, step_ms=20, spatial_center=False):
    """Load the ERS epochs of one subject, pair hits, return D and bookkeeping."""
    import mne
    pre = f"{epo_dir}/sub-{subject_id}_task-loc_desc-"
    enc = mne.read_epochs(pre + "encodingERS_epo.fif", preload=True, verbose="error").pick("eeg")
    ret = mne.read_epochs(pre + "retrievalERS_epo.fif", preload=True, verbose="error").pick("eeg")
    e_idx, r_idx, hits = pair_hits_from_epochs(enc, ret)
    D, te, tr = ers_difference(enc.get_data(), ret.get_data(), e_idx, r_idx, enc.info["sfreq"],
                               enc_tmin=enc.tmin, ret_tmin=ret.tmin, win_ms=win_ms,
                               step_ms=step_ms, spatial_center=spatial_center)
    return {"D": D, "te": te, "tr": tr, "n_hits": len(e_idx), "hits": hits}

In [ ]:
trigger_map = {
    # --- EXPERIMENT META ---
    'PLACEHOLDER': 99999,
    'psychopy_enc_starts': 1,
    'system_255': 255,  # To resolve unexpected hardware or system triggers (appeared only in sub-27, unknown source)

    # ----------------------------------
    # --- PHASE 0: PRACTICE (2-3) ---
    # ----------------------------------
    'prac_start': 2, #should appear once before practice trials, e.g., at "initialize" routine
    'prac_end': 3,   #should appear once after practice trials, e.g., at "practice end" routine. 

    # ----------------------------------
    # --- PHASE 1: ENCODING (100-199) loop ---
    # ----------------------------------
    'enc_start': 100, 

    'enc_fixation': 110, 
    'enc_cue': 101,
    'enc_act1': 102,
    'enc_act2': 103,
    'enc_act3': 104, 

    # Targets (The 2x2 Manipulation) in "target_trial" routine
    'tgt_sc_pc': 151,
    'tgt_sc_pi': 152,
    'tgt_si_pc': 153,
    'tgt_si_pi': 154,

    # Encoding Questions in "response" routine
    'enc_q_schema': 161, #Did the object reach the intended recipient?
    'enc_q_color': 162,  #Was the last panel colorized?

    # Encoding Responses (Left/Right)
    'enc_resp_left': 171,
    'enc_resp_right': 172,

    # Encoding phase ends
    'enc_end': 199,

    # ----------------------------------
    # --- PHASE 2: DISTRACTOR (50-51) ---
    # ----------------------------------
    'dist_start': 50,
    'dist_end': 51,

    # ----------------------------------
    # --- PHASE 3: RETRIEVAL (200+) ---
    # ----------------------------------
    'psychopy_ret_starts': 4,
    'ret_start': 200, 

    'ret_fixation': 210, 

    # A. Cue Display (Either 1 of 4 OLD conditions or 1 NEW condition)
    'ret_cue_old_sc_pc': 211,
    'ret_cue_old_sc_pi': 212,
    'ret_cue_old_si_pc': 213,
    'ret_cue_old_si_pi': 214,
    'ret_cue_new': 215,

    # B. Question 1: "Seen before?" (Either 1 of 4 OLD conditions or 1 NEW condition) during "cue_recog" routine
    'ret_q1_old_sc_pc': 221,
    'ret_q1_old_sc_pi': 222,
    'ret_q1_old_si_pc': 223,
    'ret_q1_old_si_pi': 224,
    'ret_q1_new': 225,

    'ret_q1_resp_left': 228,  
    'ret_q1_resp_right': 229, 

    # C. Question 2: "Unexpected?" (Either 1 of 4 OLD conditions or 1 NEW condition) during "ending_recall" routine
    'ret_q2_old_sc_pc': 231,
    'ret_q2_old_sc_pi': 232,
    'ret_q2_old_si_pc': 233,
    'ret_q2_old_si_pi': 234,
    'ret_q2_new': 235,

    'ret_q2_resp_left': 238,  
    'ret_q2_resp_right': 239, 

    # D. Target Selection (4 Choices) (Either 1 of 4 OLD conditions or 1 NEW condition) during "target_trial" routine
    'ret_q3_old_sc_pc': 241,
    'ret_q3_old_sc_pi': 242,
    'ret_q3_old_si_pc': 243,
    'ret_q3_old_si_pi': 244,
    'ret_q3_new': 245,

    'ret_q3_resp_1': 251,
    'ret_q3_resp_2': 252,
    'ret_q3_resp_3': 253,
    'ret_q3_resp_4': 254,

    'ret_end': 209
}